# 05 – SAM Fine-tuning for Finger Segmentation

Uses **Segment Anything Model (SAM)** from HuggingFace.  
Accepts binary PNG masks directly — no polygon conversion needed.

**Strategy:**
- Freeze the image encoder and prompt encoder (too large to fine-tune on 130 images)
- Fine-tune **only the mask decoder**
- For each finger mask, auto-generate a bounding-box prompt from the mask itself
- Ground truth = the original brush PNG mask

**Classes:** right-hand-index (0) · right-hand-middle (1) · right-hand-thumb (2)

## 0 · Install dependencies

In [1]:
# Run once — skip if already installed
%pip install transformers accelerate monai -q

Note: you may need to restart the kernel to use updated packages.


## 1 · Imports & paths

In [2]:
import csv
import re
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import SamModel, SamProcessor
from monai.losses import DiceLoss

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = Path(r"c:\Users\admin\Desktop\R\Projects\05_CV\Hand_Pose_Estimation")
RAW_DIR     = ROOT / "data" / "raw"          / "20260427_raw_data"
MASK_DIR    = ROOT / "data" / "annotations"  / "20260427_label"
MAPPING_CSV = ROOT / "data" / "processed"    / "20260427_brush_seg" / "task_image_mapping.csv"
CKPT_DIR    = ROOT / "runs"  / "sam_seg"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES  = ["right-hand-index", "right-hand-middle", "right-hand-thumb"]
FINGERS  = ["index", "middle", "thumb"]
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

print("Device :", DEVICE)
print("RAW_DIR:", RAW_DIR.exists(), "|", len(list(RAW_DIR.glob("*.jpg"))), "images")
print("MASKS  :", MASK_DIR.exists(), "|", len(list(MASK_DIR.glob("*.png"))), "masks")

Device : cuda
RAW_DIR: True | 130 images
MASKS  : True | 388 masks


## 2 · Load task → image mapping

In [3]:
# Read the mapping CSV produced in notebook 04
# (edit that CSV if the mapping was wrong before running this)
task_to_image: dict[int, Path] = {}
with open(MAPPING_CSV, newline="") as f:
    for row in csv.DictReader(f):
        task_to_image[int(row["task_id"])] = RAW_DIR / row["image_filename"]

print(f"Loaded {len(task_to_image)} task→image mappings")

# Build task_masks: {task_id: {finger: Path}}
mask_files = sorted(MASK_DIR.glob("*.png"))
task_masks: dict[int, dict[str, Path]] = {}
for mf in mask_files:
    m = re.match(r"task-(\d+)-annotation-\d+-by-1-tag-right-hand-(\w+)-0\.png", mf.name)
    if not m:
        continue
    tid, finger = int(m.group(1)), m.group(2)
    task_masks.setdefault(tid, {})[finger] = mf

print(f"Tasks with masks: {len(task_masks)}")

Loaded 130 task→image mappings
Tasks with masks: 130


## 3 · Dataset class

Each sample is **one finger mask** from one task.  
The bounding box of the mask is used as the SAM prompt so the model knows roughly where to look.

In [4]:
def mask_to_bbox(mask_arr: np.ndarray) -> list[int]:
    """
    Returns [x_min, y_min, x_max, y_max] of the foreground region.
    Used as the bounding-box prompt fed to SAM.
    """
    rows = np.any(mask_arr, axis=1)
    cols = np.any(mask_arr, axis=0)
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]
    # Add small padding so SAM isn't squeezed to the exact edge
    pad = 5
    h, w = mask_arr.shape
    return [
        max(0, x_min - pad), max(0, y_min - pad),
        min(w, x_max + pad), min(h, y_max + pad),
    ]


class FingerMaskDataset(Dataset):
    """
    One item = one (image, finger_mask, bbox_prompt, class_id) tuple.
    Each task contributes up to 3 items (one per finger).
    """
    def __init__(self, task_ids: list[int], processor: SamProcessor):
        self.processor = processor
        self.samples: list[dict] = []

        for tid in task_ids:
            img_path = task_to_image[tid]
            for finger, mask_path in task_masks[tid].items():
                self.samples.append({
                    "image_path" : img_path,
                    "mask_path"  : mask_path,
                    "class_id"   : FINGERS.index(finger),
                    "finger"     : finger,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # ── Raw image (RGB) ───────────────────────────────────────────────────
        image = Image.open(s["image_path"]).convert("RGB")

        # ── Binary mask (0 or 1) ──────────────────────────────────────────────
        mask_arr = np.array(Image.open(s["mask_path"]).convert("L"))
        mask_bin = (mask_arr > 10).astype(np.uint8)   # 0 or 1

        # ── Bounding-box prompt ───────────────────────────────────────────────
        if mask_bin.max() == 0:                        # empty mask — skip with dummy box
            bbox = [0, 0, image.width, image.height]
        else:
            bbox = mask_to_bbox(mask_bin)

        # ── SAM processor: resize image, encode prompt ────────────────────────
        inputs = self.processor(
            images          = image,
            input_boxes     = [[bbox]],   # shape: [[x0,y0,x1,y1]]
            return_tensors  = "pt",
        )
        # Remove batch dim added by processor
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        # ── Ground-truth mask resized to 256×256 (SAM's output resolution) ───
        gt_mask = torch.tensor(
            cv2.resize(mask_bin, (256, 256), interpolation=cv2.INTER_NEAREST),
            dtype=torch.float32
        ).unsqueeze(0)   # (1, 256, 256)

        inputs["ground_truth_mask"] = gt_mask
        inputs["class_id"]          = torch.tensor(s["class_id"])
        return inputs


print("FingerMaskDataset defined.")

FingerMaskDataset defined.


## 4 · Train / val split & DataLoaders

In [ ]:
random.seed(42)

all_tids   = sorted(task_masks.keys())
random.shuffle(all_tids)
split_idx  = int(len(all_tids) * 0.8)
train_tids = all_tids[:split_idx]
val_tids   = all_tids[split_idx:]
print(f"Train tasks: {len(train_tids)}  |  Val tasks: {len(val_tids)}")

# Load SAM processor (handles image resizing and prompt encoding)
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

train_ds = FingerMaskDataset(train_tids, processor)
val_ds   = FingerMaskDataset(val_tids,   processor)
print(f"Train samples: {len(train_ds)}  |  Val samples: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=0)

## 5 · Load SAM & freeze encoder

Only the **mask decoder** is trained.  
The image encoder has ~89 M parameters — too large to fine-tune on 130 images.

In [ ]:
model = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE)

# Freeze image encoder and prompt encoder
for param in model.vision_encoder.parameters():
    param.requires_grad = False
for param in model.prompt_encoder.parameters():
    param.requires_grad = False

# Only mask_decoder is trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.1f}% of total {total:,})")

## 6 · Training loop

Loss = **Dice loss + Binary cross-entropy** (standard for segmentation masks).

In [ ]:
import torch.nn.functional as F

optimizer  = torch.optim.AdamW(model.mask_decoder.parameters(), lr=1e-4, weight_decay=1e-4)
dice_loss  = DiceLoss(sigmoid=True, reduction="mean")
NUM_EPOCHS = 30
best_val   = float("inf")

train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0

    for batch in train_loader:
        pixel_values      = batch["pixel_values"].to(DEVICE)
        input_boxes       = batch["input_boxes"].to(DEVICE)
        gt_mask           = batch["ground_truth_mask"].to(DEVICE)  # (B,1,256,256)

        # Forward pass
        outputs = model(
            pixel_values      = pixel_values,
            input_boxes       = input_boxes,
            multimask_output  = False,   # single mask per prompt
        )
        # predicted_masks shape: (B, 1, 256, 256)  — logits (before sigmoid)
        pred_masks = outputs.pred_masks.squeeze(1)   # (B, 1, 256, 256)

        # Dice loss + BCE loss
        loss = dice_loss(pred_masks, gt_mask) + \
               F.binary_cross_entropy_with_logits(pred_masks, gt_mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(DEVICE)
            input_boxes  = batch["input_boxes"].to(DEVICE)
            gt_mask      = batch["ground_truth_mask"].to(DEVICE)

            outputs    = model(pixel_values=pixel_values,
                               input_boxes=input_boxes,
                               multimask_output=False)
            pred_masks = outputs.pred_masks.squeeze(1)
            loss       = dice_loss(pred_masks, gt_mask) + \
                         F.binary_cross_entropy_with_logits(pred_masks, gt_mask)
            val_loss  += loss.item()

    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), CKPT_DIR / "best.pt")
        tag = "  ← best"
    else:
        tag = ""

    print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}{tag}")

print("\nTraining complete. Best val loss:", round(best_val, 4))

## 7 · Loss curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train")
plt.plot(val_losses,   label="Val")
plt.xlabel("Epoch")
plt.ylabel("Dice + BCE loss")
plt.title("SAM fine-tuning – loss curve")
plt.legend()
plt.tight_layout()
plt.show()

## 8 · Evaluate on val set – visualise predictions

Loads `best.pt` and shows predicted mask vs ground truth for 6 random val samples.

In [ ]:
FINGER_COLORS = {
    "index":  np.array([255,  60,  60]),
    "middle": np.array([ 60, 220,  60]),
    "thumb":  np.array([ 60,  60, 255]),
}

# Load best checkpoint
model.load_state_dict(torch.load(CKPT_DIR / "best.pt", map_location=DEVICE))
model.eval()

samples = random.sample(val_ds.samples, min(6, len(val_ds.samples)))

fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))

for row, s in enumerate(samples):
    image    = Image.open(s["image_path"]).convert("RGB")
    mask_arr = np.array(Image.open(s["mask_path"]).convert("L"))
    mask_bin = (mask_arr > 10).astype(np.uint8)
    finger   = s["finger"]
    color    = FINGER_COLORS[finger]

    bbox   = mask_to_bbox(mask_bin) if mask_bin.max() > 0 else [0, 0, image.width, image.height]
    inputs = processor(images=image, input_boxes=[[bbox]], return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out  = model(**inputs, multimask_output=False)
        pred = torch.sigmoid(out.pred_masks.squeeze()).cpu().numpy()  # (256,256)

    pred_bin = (pred > 0.5).astype(np.uint8)
    pred_up  = cv2.resize(pred_bin, (image.width, image.height), interpolation=cv2.INTER_NEAREST)

    img_np   = np.array(image)
    overlay  = img_np.copy()
    overlay[pred_up > 0] = (img_np[pred_up > 0] * 0.5 + color * 0.5).astype(np.uint8)

    # Draw bbox
    x0, y0, x1, y1 = bbox
    cv2.rectangle(overlay, (x0, y0), (x1, y1), color.tolist(), 2)

    ax_img, ax_gt, ax_pred = axes[row]
    ax_img.imshow(img_np);              ax_img.set_title(f"{finger}\n{s['image_path'].name}", fontsize=7)
    ax_gt.imshow(mask_bin, cmap="gray"); ax_gt.set_title("Ground truth", fontsize=7)
    ax_pred.imshow(overlay);            ax_pred.set_title("Prediction", fontsize=7)
    for ax in (ax_img, ax_gt, ax_pred):
        ax.axis("off")

plt.suptitle("SAM predictions vs ground truth (val set)", fontsize=12)
plt.tight_layout()
plt.show()

## 9 · Dice score per class on val set

In [ ]:
from collections import defaultdict

def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    intersection = (pred * gt).sum()
    return (2 * intersection) / (pred.sum() + gt.sum() + 1e-6)

scores_per_class = defaultdict(list)

model.eval()
for s in val_ds.samples:
    image    = Image.open(s["image_path"]).convert("RGB")
    mask_arr = np.array(Image.open(s["mask_path"]).convert("L"))
    mask_bin = (mask_arr > 10).astype(np.uint8)
    finger   = s["finger"]

    if mask_bin.max() == 0:
        continue

    bbox   = mask_to_bbox(mask_bin)
    inputs = processor(images=image, input_boxes=[[bbox]], return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out  = model(**inputs, multimask_output=False)
        pred = torch.sigmoid(out.pred_masks.squeeze()).cpu().numpy()

    pred_bin = (pred > 0.5).astype(np.uint8)
    pred_up  = cv2.resize(pred_bin, (image.width, image.height), interpolation=cv2.INTER_NEAREST)
    scores_per_class[finger].append(dice_score(pred_up, mask_bin))

print("Dice score per finger (val set):")
for finger in FINGERS:
    sc = scores_per_class.get(finger, [])
    print(f"  {finger:8s}: {np.mean(sc):.4f}  (n={len(sc)})")